# CER — Cluster Experiment Runner Demo

This notebook demonstrates the full CER workflow:

1. **Build** the Singularity container locally
2. **Run** the agent inside the container
3. **Submit** experiments to the cluster via MCP
4. **Monitor** job status and query W&B results

Architecture:
- The experiment code and agent run **inside** a Singularity/Apptainer container
- `cer-mcp` runs on the **host** with SSH access to the cluster
- The agent connects to the MCP server over `localhost` (Apptainer shares host networking)

## 0. Build the Container

The container is defined in `experiment.def` and provides the full runtime environment
(PyTorch, W&B, your pip dependencies). The same container runs locally and on the cluster.

**Prerequisites:** [Apptainer](https://apptainer.org/docs/admin/main/installation.html) installed on your machine.

In [ ]:
# Check Apptainer is installed
!apptainer --version

In [ ]:
# Build the container from the definition file
# This pulls the base image and installs dependencies (~10-15 min first time, ~9GB)
!apptainer build experiment.sif experiment.def

In [ ]:
# Verify the container works
!apptainer exec experiment.sif python -c "import torch; print(f'PyTorch {torch.__version__}')"

### Running inside the container

Use `apptainer exec` or `apptainer shell` to work inside the container:

```bash
# Run a command
apptainer exec --nv experiment.sif python train.py

# Interactive shell (for the agent or manual work)
apptainer exec --nv --bind .:/workspace --pwd /workspace experiment.sif bash

# With GPU (--nv) and workspace mounted:
#   - Your code is at /workspace
#   - Host networking is shared (agent can reach cer-mcp on localhost)
#   - No SSH access inside the container
```

**Rebuild** only when `experiment.def` changes (new system/pip deps). Code changes don't require a rebuild — the workspace is bind-mounted.

## 1. Start the MCP Server

In a separate terminal on the **host** (outside the container), run:
```bash
cer-mcp
```

This starts the server on `http://localhost:8000/sse`.

Inside the container, the `./cer` CLI client connects to it over localhost.

In [ ]:
import subprocess

CER = "./cer"

# Check it works
!{CER} --help

## 2. Commit & Push

CER uses **commit = experiment**. Every experiment is tied to a git commit for full reproducibility.

In [ ]:
!git status
print("---")
!git log --oneline -5

In [ ]:
# Uncomment to commit and push:
# !git add -A && git commit -m "experiment: describe changes" && git push

In [ ]:
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Commit: {COMMIT}")

## 3. Submit Experiment

Calls `submit` tool on the MCP server → SSH to cluster → create worktree → sbatch.

In [ ]:
!{CER} submit {COMMIT}

In [ ]:
JOB_ID = input("Enter job ID from above: ")

## 4. Check Status

Queries SLURM via SSH. Run multiple times to watch: `PENDING` → `RUNNING` → `COMPLETED`.

In [ ]:
!{CER} status {JOB_ID}

## 5. List All Experiments

Returns all tracked experiments. Active jobs get their SLURM status refreshed.

In [ ]:
!{CER} list

## 6. Query W&B Results

Finds the W&B run by commit hash tag, pulls summary and history.

In [ ]:
# Summary metrics
!{CER} results {JOB_ID}

In [ ]:
# Full training history
!{CER} results {JOB_ID} --history

In [ ]:
# Filtered metrics
!{CER} results {JOB_ID} --history --key train/loss --key val/loss

## 7. Programmatic Access (JSON)

Parse the JSON output for plotting or automated pipelines.

In [ ]:
import json

raw = subprocess.check_output([CER, "results", JOB_ID, "--history"]).decode()
data = json.loads(raw)

print(f"Run:     {data['wandb']['run_name']} ({data['wandb']['state']})")
print(f"URL:     {data['wandb']['url']}")
print(f"Summary: {data['wandb']['summary']}")
print(f"History: {len(data.get('history', []))} steps")

## 8. Plot Results

In [ ]:
import matplotlib.pyplot as plt

history = data["history"]
steps = range(len(history))
train_loss = [h.get("train/loss") for h in history]
val_loss = [h.get("val/loss") for h in history]

plt.figure(figsize=(10, 5))
if any(v is not None for v in train_loss):
    plt.plot(steps, train_loss, label="train/loss", marker="o")
if any(v is not None for v in val_loss):
    plt.plot(steps, val_loss, label="val/loss", marker="s")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title(f"Experiment {JOB_ID} (commit {COMMIT[:8]})")
plt.legend()
plt.grid(True)
plt.show()

## 9. Cancel an Experiment

In [ ]:
# Uncomment to cancel a running job:
# !{CER} cancel {JOB_ID}

---

## Command Reference

The `./cer` CLI client connects to the MCP server (`cer-mcp`) running on the host.

| Command | Description |
|---------|-------------|
| `./cer submit <commit>` | Submit experiment to SLURM cluster |
| `./cer status <job_id>` | Check SLURM job status (JSON) |
| `./cer cancel <job_id>` | Cancel a running job |
| `./cer results <job_id>` | W&B summary metrics (JSON) |
| `./cer results <job_id> --history` | Full metric history |
| `./cer results <job_id> --key <metric>` | Filter specific metrics |
| `./cer list` | List all experiments |